# Compare Two CSV Files with Polars

This notebook compares two CSV files and reports:
- whether they are identical
- column differences (names / dtypes / column-only-in-one-file)
- row count differences
- cell-level differences (row index or key, column, value in each file)

**Row-alignment strategies (Section 5):**
- **Positional** (default): row 0 vs row 0, row 1 vs row 1, etc. Assumes same order.
- **Key-based**: match rows by one or more key columns (handles reordered/inserted/deleted rows).
- **Sort-based**: sort both files the same way first, then compare positionally. A lighter alternative to a key when there's no unique identifier.

**Additional comparison methods (Section 8):**
- Tolerance-based numeric comparison (ignore tiny floating-point noise)
- Row-set / hash comparison (order- and duplicate-agnostic, no key needed)
- Schema-only comparison (columns/dtypes, no data)
- Summary-statistics comparison (per-column aggregates)
- Duplicate-row detection within each file
- String normalization before comparing (trim/case-fold)
- Column-order-only difference detection
- Percentage-based summary diff report

## 1. Setup

In [ ]:
import polars as pl


## 2. Parameters

Edit these values, then run all cells below.

In [ ]:
# --- Edit these ---
FILE1 = "file1.csv"
FILE2 = "file2.csv"

# Optional: match rows by key column(s) instead of row position.
# Set to None to compare by position, or a list like ["id"] or ["id", "region"].
KEY_COLS = None

# Optional: sort both dataframes before a positional comparison (alternative to KEY_COLS).
# Set to None to skip sorting, or a list like ["id"], or [] to sort by ALL common columns.
SORT_COLS = None

# Optional: path to save the cell-level differences as a CSV. Set to None to skip saving.
OUTPUT_PATH = None

# Optional: relative/absolute tolerance for numeric columns (Section 8.1).
# Set to None to require exact equality.
NUMERIC_RTOL = 1e-9
NUMERIC_ATOL = 1e-9


## 3. Load the CSV files

In [ ]:
def load_csv(path: str) -> pl.DataFrame:
    try:
        return pl.read_csv(path, infer_schema_length=10000)
    except Exception as e:
        raise RuntimeError(f"could not read '{path}': {e}")


df1 = load_csv(FILE1)
df2 = load_csv(FILE2)

print(f"file1: {FILE1}  -> {df1.height} rows x {df1.width} cols")
print(f"file2: {FILE2}  -> {df2.height} rows x {df2.width} cols")


In [ ]:
df1.head()


In [ ]:
df2.head()


## 4. Core helper functions

In [ ]:
def sort_dataframe(df: pl.DataFrame, sort_cols=None) -> pl.DataFrame:
    """
    Return a sorted copy of df.

    - sort_cols: list of column names to sort by. If None or empty,
      sorts by every column in the dataframe (in their existing order).
    - nulls are placed last so they don't jumble ordering unpredictably.
    """
    cols = sort_cols if sort_cols else df.columns
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"sort column(s) not found in dataframe: {missing}")
    return df.sort(by=cols, nulls_last=True)


In [ ]:
def compare_columns(df1: pl.DataFrame, df2: pl.DataFrame):
    cols1, cols2 = set(df1.columns), set(df2.columns)
    only_in_1 = cols1 - cols2
    only_in_2 = cols2 - cols1
    common = cols1 & cols2

    dtype_diffs = []
    for c in sorted(common):
        t1, t2 = df1.schema[c], df2.schema[c]
        if t1 != t2:
            dtype_diffs.append((c, t1, t2))

    return only_in_1, only_in_2, common, dtype_diffs


In [ ]:
def values_differ(s1: pl.Series, s2: pl.Series, rtol=None, atol=None) -> pl.Series:
    """
    Null-safe element-wise inequality mask between two series of the same length.

    - Treats null == null as equal (not a difference).
    - If rtol/atol are given AND both series are numeric, uses a tolerance-based
      comparison (like numpy.isclose) instead of exact equality, so tiny floating
      point noise (e.g. 0.1 + 0.2 != 0.3) is not reported as a difference.
    """
    both_numeric = s1.dtype.is_numeric() and s2.dtype.is_numeric()

    if rtol is not None and atol is not None and both_numeric:
        diff = (s1 - s2).abs()
        threshold = atol + rtol * s2.abs()
        close = diff <= threshold
        neq_mask = ~close
    else:
        neq_mask = ~(s1 == s2)

    neq_mask = neq_mask | (s1.is_null() != s2.is_null())
    return neq_mask.fill_null(True)


In [ ]:
def compare_by_position(df1: pl.DataFrame, df2: pl.DataFrame, common_cols, rtol=None, atol=None):
    """Compare rows purely by row index (order matters)."""
    n1, n2 = df1.height, df2.height
    n = min(n1, n2)

    diffs = []
    if n > 0:
        sub1 = df1.select(sorted(common_cols)).head(n)
        sub2 = df2.select(sorted(common_cols)).head(n)

        for col in sorted(common_cols):
            s1 = sub1[col]
            s2 = sub2[col]
            neq_mask = values_differ(s1, s2, rtol=rtol, atol=atol)

            if neq_mask.any():
                idx = [i for i, v in enumerate(neq_mask) if v]
                v1 = s1.to_list()
                v2 = s2.to_list()
                for i in idx:
                    diffs.append(
                        {
                            "row_index": i,
                            "column": col,
                            "value_file1": v1[i],
                            "value_file2": v2[i],
                        }
                    )

    extra_rows_1 = n1 - n if n1 > n2 else 0
    extra_rows_2 = n2 - n if n2 > n1 else 0
    return diffs, extra_rows_1, extra_rows_2


In [ ]:
def compare_by_key(df1: pl.DataFrame, df2: pl.DataFrame, common_cols, key_cols, rtol=None, atol=None):
    """Compare rows matched by key column(s), robust to reordering/inserts/deletes."""
    for k in key_cols:
        if k not in common_cols:
            raise ValueError(f"key column '{k}' not present (as common column) in both files.")

    value_cols = sorted(c for c in common_cols if c not in key_cols)

    d1 = df1.select(key_cols + value_cols)
    d2 = df2.select(key_cols + value_cols)

    dup1 = d1.filter(pl.struct(key_cols).is_duplicated())
    dup2 = d2.filter(pl.struct(key_cols).is_duplicated())
    if dup1.height or dup2.height:
        print("WARNING: duplicate key values found; comparison may be unreliable.")
        if dup1.height:
            print(f"  file1 duplicate keys:\n{dup1.select(key_cols)}")
        if dup2.height:
            print(f"  file2 duplicate keys:\n{dup2.select(key_cols)}")

    only_1 = d1.join(d2, on=key_cols, how="anti")
    only_2 = d2.join(d1, on=key_cols, how="anti")

    both = d1.join(d2, on=key_cols, how="inner", suffix="_f2")

    diffs = []
    if both.height and value_cols:
        for col in value_cols:
            s1 = both[col]
            s2 = both[f"{col}_f2"]
            neq_mask = values_differ(s1, s2, rtol=rtol, atol=atol)

            if neq_mask.any():
                idxs = [i for i, v in enumerate(neq_mask) if v]
                keys_vals = both.select(key_cols)
                v1 = s1.to_list()
                v2 = s2.to_list()
                for i in idxs:
                    row_key = {k: keys_vals[k][i] for k in key_cols}
                    diffs.append(
                        {
                            **row_key,
                            "column": col,
                            "value_file1": v1[i],
                            "value_file2": v2[i],
                        }
                    )

    return diffs, only_1, only_2


## 5. Run the main comparison

In [ ]:
if KEY_COLS and SORT_COLS is not None:
    raise ValueError("use either KEY_COLS or SORT_COLS, not both.")

diff_df = None
only_rows_1 = only_rows_2 = None

print(f"file1: {FILE1}  -> {df1.height} rows x {df1.width} cols")
print(f"file2: {FILE2}  -> {df2.height} rows x {df2.width} cols")
print("-" * 60)

if df1.equals(df2):
    print("RESULT: The two CSV files are IDENTICAL (same columns, order, and values).")
else:
    only_in_1, only_in_2, common, dtype_diffs = compare_columns(df1, df2)

    if only_in_1:
        print(f"Columns only in file1: {sorted(only_in_1)}")
    if only_in_2:
        print(f"Columns only in file2: {sorted(only_in_2)}")
    if dtype_diffs:
        print("Columns with differing dtypes:")
        for c, t1, t2 in dtype_diffs:
            print(f"  - {c}: file1={t1}  file2={t2}")

    if not common:
        print("No common columns to compare row/cell values.")
    else:
        diffs = []

        if KEY_COLS:
            diffs, only_rows_1, only_rows_2 = compare_by_key(
                df1, df2, common, KEY_COLS, rtol=NUMERIC_RTOL, atol=NUMERIC_ATOL
            )
            if only_rows_1.height:
                print(f"\nRows only in file1 (by key {KEY_COLS}): {only_rows_1.height}")
            if only_rows_2.height:
                print(f"\nRows only in file2 (by key {KEY_COLS}): {only_rows_2.height}")

        else:
            cmp_df1, cmp_df2 = df1, df2
            if SORT_COLS is not None:
                sort_cols = list(SORT_COLS) if SORT_COLS else sorted(common)
                print(f"\nSorting both dataframes by: {sort_cols}")
                cmp_df1 = sort_dataframe(df1, sort_cols)
                cmp_df2 = sort_dataframe(df2, sort_cols)

            diffs, extra_1, extra_2 = compare_by_position(
                cmp_df1, cmp_df2, common, rtol=NUMERIC_RTOL, atol=NUMERIC_ATOL
            )
            if extra_1:
                print(f"\nfile1 has {extra_1} extra trailing row(s) not present in file2.")
            if extra_2:
                print(f"\nfile2 has {extra_2} extra trailing row(s) not present in file1.")

        if diffs:
            diff_df = pl.DataFrame(diffs)
            print(f"\nCell-level differences found: {len(diffs)}")
        else:
            print("\nNo cell-level differences found among common rows/columns.")

        print("-" * 60)
        print("RESULT: The two CSV files are DIFFERENT.")


## 6. Inspect the differences

In [ ]:
if diff_df is not None:
    display(diff_df)
else:
    print("No cell-level diff table to show.")


In [ ]:
if only_rows_1 is not None and only_rows_1.height:
    print("Rows only in file1:")
    display(only_rows_1)
if only_rows_2 is not None and only_rows_2.height:
    print("Rows only in file2:")
    display(only_rows_2)
if (only_rows_1 is None or not only_rows_1.height) and (only_rows_2 is None or not only_rows_2.height):
    print("KEY_COLS was not used (or no key-only rows found).")


## 7. Save the differences (optional)

In [ ]:
if OUTPUT_PATH and diff_df is not None:
    diff_df.write_csv(OUTPUT_PATH)
    print(f"Saved full diff report to: {OUTPUT_PATH}")
else:
    print("Nothing saved (either OUTPUT_PATH is None or there were no diffs).")


## 8. Additional comparison methods

These are independent, optional checks — run whichever cells are useful for your data. They don't depend on the row-alignment choice made in Section 5.

### 8.1 Tolerance-based numeric comparison (already wired in above)

In [ ]:
# Tolerance is already applied in Section 5 via NUMERIC_RTOL / NUMERIC_ATOL, using the
# same logic as numpy.isclose: |a - b| <= atol + rtol * |b|.
# To compare with EXACT equality instead, set NUMERIC_RTOL = NUMERIC_ATOL = None in
# Section 2 and re-run Section 5.

print(f"Current tolerance: rtol={NUMERIC_RTOL}, atol={NUMERIC_ATOL}")
print("Set both to None in Section 2 for exact equality, then re-run Section 5.")


### 8.2 Row-set / hash comparison

Order- and duplicate-agnostic: hashes each full row and compares the *sets* of hashes.
Tells you which rows exist in one file but not the other, without needing a key column.

Caveat: an edited row looks like one row removed + one row added — you get "rows only
in file1/file2", not a cell-level diff. Use `compare_by_key` if you need that detail.

In [ ]:
def compare_row_sets(df1: pl.DataFrame, df2: pl.DataFrame, common_cols):
    """
    Compare two dataframes as sets of rows (ignoring order and row position).
    Returns (rows_only_in_1, rows_only_in_2, n_common_rows).
    """
    cols = sorted(common_cols)
    d1 = df1.select(cols).with_columns(pl.struct(cols).hash().alias("_row_hash"))
    d2 = df2.select(cols).with_columns(pl.struct(cols).hash().alias("_row_hash"))

    hashes1 = set(d1["_row_hash"].to_list())
    hashes2 = set(d2["_row_hash"].to_list())

    only_hashes_1 = hashes1 - hashes2
    only_hashes_2 = hashes2 - hashes1
    common_hashes = hashes1 & hashes2

    rows_only_1 = d1.filter(pl.col("_row_hash").is_in(only_hashes_1)).drop("_row_hash")
    rows_only_2 = d2.filter(pl.col("_row_hash").is_in(only_hashes_2)).drop("_row_hash")

    return rows_only_1, rows_only_2, len(common_hashes)


try:
    common_cols_for_rowset = common
except NameError:
    _, _, common_cols_for_rowset, _ = compare_columns(df1, df2)

rowset_only_1, rowset_only_2, n_common_rows = compare_row_sets(df1, df2, common_cols_for_rowset)

print(f"Rows identical in both files (as a set): {n_common_rows}")
print(f"Rows only in file1 (row-set comparison): {rowset_only_1.height}")
print(f"Rows only in file2 (row-set comparison): {rowset_only_2.height}")

if rowset_only_1.height:
    display(rowset_only_1)
if rowset_only_2.height:
    display(rowset_only_2)


### 8.3 Schema-only comparison

Compares just column names, order, and dtypes — no data touched. Useful as a fast
pre-check, e.g. validating that an ETL job didn't change the schema.

In [ ]:
def compare_schema_only(df1: pl.DataFrame, df2: pl.DataFrame):
    same_names_and_order = df1.columns == df2.columns
    same_dtypes = df1.schema == df2.schema
    return {
        "same_column_names_and_order": same_names_and_order,
        "same_names_ignoring_order": set(df1.columns) == set(df2.columns),
        "same_dtypes_for_shared_columns": same_dtypes,
        "file1_schema": dict(df1.schema),
        "file2_schema": dict(df2.schema),
    }


schema_result = compare_schema_only(df1, df2)
print(f"Same column names AND order: {schema_result['same_column_names_and_order']}")
print(f"Same column names (order ignored): {schema_result['same_names_ignoring_order']}")
print(f"Identical full schema (names+order+dtypes): {schema_result['same_dtypes_for_shared_columns']}")


### 8.4 Column-order-only difference

Flags the specific case where both files have the exact same columns and same data,
just arranged in a different column order — which `df1.equals(df2)` would otherwise
just call "different".

In [ ]:
def is_column_order_only_diff(df1: pl.DataFrame, df2: pl.DataFrame) -> bool:
    """True if df1 and df2 have identical data & dtypes, differing only in column order."""
    if set(df1.columns) != set(df2.columns):
        return False
    if df1.columns == df2.columns:
        return False  # same order -> not this case (either identical or a real diff)
    return df1.select(sorted(df1.columns)).equals(df2.select(sorted(df2.columns)))


if is_column_order_only_diff(df1, df2):
    print("Files contain identical data — the only difference is column order.")
else:
    print("Not a pure column-order difference (either identical, or data actually differs).")


### 8.5 Summary-statistics comparison

For large files, a full cell-by-cell diff can be overkill. This compares per-column
aggregates instead: row count, null count, and (for numeric columns) min/max/mean/std,
or (for string columns) number of unique values.

In [ ]:
def summarize_column(df: pl.DataFrame, col: str) -> dict:
    s = df[col]
    summary = {
        "n_rows": df.height,
        "n_nulls": s.null_count(),
        "n_unique": s.n_unique(),
    }
    if s.dtype.is_numeric():
        summary.update({
            "min": s.min(),
            "max": s.max(),
            "mean": s.mean(),
            "std": s.std(),
        })
    return summary


def compare_summary_stats(df1: pl.DataFrame, df2: pl.DataFrame, common_cols):
    rows = []
    for col in sorted(common_cols):
        s1 = summarize_column(df1, col)
        s2 = summarize_column(df2, col)
        row = {"column": col}
        for k in s1:
            row[f"{k}_file1"] = s1[k]
            row[f"{k}_file2"] = s2.get(k)
        rows.append(row)
    return pl.DataFrame(rows)


try:
    common_cols_for_summary = common
except NameError:
    _, _, common_cols_for_summary, _ = compare_columns(df1, df2)

summary_df = compare_summary_stats(df1, df2, common_cols_for_summary)
display(summary_df)


### 8.6 Duplicate-row detection within each file

Flags rows that are duplicated *within* a single file (independent of the other file).

In [ ]:
def find_duplicate_rows(df: pl.DataFrame, subset_cols=None) -> pl.DataFrame:
    """Return all rows that are duplicated within df (all occurrences, not just extras)."""
    cols = subset_cols if subset_cols else df.columns
    return df.filter(pl.struct(cols).is_duplicated())


dupes_1 = find_duplicate_rows(df1)
dupes_2 = find_duplicate_rows(df2)

print(f"Duplicate rows within file1: {dupes_1.height}")
print(f"Duplicate rows within file2: {dupes_2.height}")

if dupes_1.height:
    display(dupes_1)
if dupes_2.height:
    display(dupes_2)


### 8.7 String normalization before comparing

Trims whitespace and case-folds string columns before comparing, so `"Foo "` vs `"foo"`
isn't reported as a difference. Returns normalized copies — use these in place of
`df1`/`df2` with any of the comparison functions above if you want this behavior.

In [ ]:
def normalize_strings(df: pl.DataFrame, case_insensitive=True, strip_whitespace=True) -> pl.DataFrame:
    """Return a copy of df with string columns trimmed and optionally lowercased."""
    exprs = []
    for col, dtype in df.schema.items():
        if dtype == pl.Utf8:
            e = pl.col(col)
            if strip_whitespace:
                e = e.str.strip_chars()
            if case_insensitive:
                e = e.str.to_lowercase()
            exprs.append(e.alias(col))
        else:
            exprs.append(pl.col(col))
    return df.select(exprs)


df1_normalized = normalize_strings(df1)
df2_normalized = normalize_strings(df2)

print("Created df1_normalized / df2_normalized (whitespace-trimmed, lowercased strings).")
print("Pass these into compare_by_position / compare_by_key / compare_row_sets instead")
print("of df1 / df2 if you want normalization applied.")

# Example: does normalization alone explain all the differences?
if df1_normalized.equals(df2_normalized):
    print("\nAfter normalization, the files are IDENTICAL — differences were just whitespace/case.")
else:
    print("\nFiles still differ after string normalization.")


### 8.8 Percentage-based summary diff report

For a high-level view when there are many diffs: percent of rows matching, percent of
cells matching per column, and which columns have the most mismatches. Requires
`compare_by_position` or `compare_by_key` to have been run in Section 5 (uses `diff_df`).

In [ ]:
def summarize_diff_percentages(diff_df: pl.DataFrame, n_rows_compared: int) -> pl.DataFrame:
    """
    Given a diff_df with a 'column' field (as produced by compare_by_position /
    compare_by_key) and the number of rows that were compared, report per-column
    mismatch counts and percentages.
    """
    if diff_df is None or diff_df.height == 0 or n_rows_compared == 0:
        return pl.DataFrame({"column": [], "n_mismatches": [], "pct_mismatched": []})

    return (
        diff_df.group_by("column")
        .agg(pl.len().alias("n_mismatches"))
        .with_columns(
            (pl.col("n_mismatches") / n_rows_compared * 100).round(2).alias("pct_mismatched")
        )
        .sort("n_mismatches", descending=True)
    )


if diff_df is not None:
    n_compared = min(df1.height, df2.height) if not KEY_COLS else max(df1.height, df2.height)
    pct_report = summarize_diff_percentages(diff_df, n_compared)
    print(f"Rows compared: {n_compared}")
    display(pct_report)
else:
    print("No diff_df available — run Section 5 first (and make sure differences were found).")
